In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import geopandas as gpd
from shapely import wkt
from shapely.geometry import Point

In [3]:
df = pd.read_csv('/content/drive/MyDrive/final_csv/원본 파일/Chicago_Crimes_2005_to_2007.csv', on_bad_lines='skip')

In [4]:
df = df.dropna(subset=['X Coordinate','Y Coordinate','Latitude', 'Longitude', 'Location'])

In [5]:
df.isna().sum()

,0
Unnamed: 0,0
ID,0
Case Number,0
Date,0
Block,0
IUCR,0
Primary Type,0
Description,0
Location Description,14
Arrest,0


In [6]:
# 3. 범죄 데이터의 위도, 경도 정보를 사용해 GeoDataFrame 생성
df['geometry'] = df.apply(lambda row: Point(row['Longitude'], row['Latitude']), axis=1)
crime_gdf = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")  # WGS84 좌표계 설정

In [7]:
# 4. Community Area CSV 파일 불러오기 (the_geom을 WKT 형식으로 변환)
community_areas = pd.read_csv('/content/drive/MyDrive/final_csv/CommAreas_20250325.csv')  # Community Area 데이터 (CSV)
community_areas['geometry'] = community_areas['the_geom'].apply(wkt.loads)  # the_geom을 WKT로 변환
community_areas_gdf = gpd.GeoDataFrame(community_areas, geometry='geometry', crs="EPSG:4326")

In [ ]:
community_areas_gdf.head()

,the_geom,PERIMETER,AREA,COMAREA_,COMAREA_ID,AREA_NUMBE,COMMUNITY,AREA_NUM_1,SHAPE_AREA,SHAPE_LEN,geometry
0,MULTIPOLYGON (((-87.60914087617894 41.84469250...,0,0,0,0,35,DOUGLAS,35,4.600462e+07,31027.054510,"MULTIPOLYGON (((-87.60914 41.84469, -87.60915 ..."
1,MULTIPOLYGON (((-87.59215283879394 41.81692934...,0,0,0,0,36,OAKLAND,36,1.691396e+07,19565.506153,"MULTIPOLYGON (((-87.59215 41.81693, -87.59231 ..."
2,MULTIPOLYGON (((-87.62879823733725 41.80189303...,0,0,0,0,37,FULLER PARK,37,1.991670e+07,25339.089750,"MULTIPOLYGON (((-87.6288 41.80189, -87.62879 4..."
3,MULTIPOLYGON (((-87.6067081256125 41.816813770...,0,0,0,0,38,GRAND BOULEVARD,38,4.849250e+07,28196.837157,"MULTIPOLYGON (((-87.60671 41.81681, -87.6067 4..."
4,MULTIPOLYGON (((-87.59215283879394 41.81692934...,0,0,0,0,39,KENWOOD,39,2.907174e+07,23325.167906,"MULTIPOLYGON (((-87.59215 41.81693, -87.59215 ..."


In [8]:
# 5. Ward CSV 파일 불러오기 (the_geom을 WKT 형식으로 변환)
wards = pd.read_csv('/content/drive/MyDrive/final_csv/Wards_2003_2015.csv')  # Ward 데이터 (CSV)
wards['geometry'] = wards['the_geom'].apply(wkt.loads)  # the_geom을 WKT로 변환
wards_gdf = gpd.GeoDataFrame(wards, geometry='geometry', crs="EPSG:4326")

In [9]:
wards_gdf = wards_gdf[wards_gdf['WARD'] != 'OUT']

In [10]:
wards_gdf= wards_gdf.dropna(subset=['WARD'])

# int로 변환 후 float로 변환
wards_gdf['WARD'] = wards_gdf['WARD'].astype(float)

In [11]:
wards_gdf['WARD'].unique()

array([ 4., 33., 49., 37., 18., 31., 25.,  8., 26., 28.,  3., 47.,  1.,
       38., 11., 30., 39., 12.,  9.,  6.,  5., 19., 41., 23., 24., 46.,
       44., 36., 48., 27., 50.,  7., 15., 34., 40., 10.,  2., 22., 35.,
       32., 17., 21., 16., 45., 42., 13., 14., 43., 29., 20.])

In [12]:
# 6. 범죄 데이터의 좌표가 어느 Ward에 속하는지 Spatial Join
crime_with_ward = gpd.sjoin(crime_gdf, wards_gdf[['WARD', 'geometry']], how='left', predicate='within')  # Ward 매핑

In [13]:
# 7. 범죄 데이터의 좌표가 어느 Community Area에 속하는지 Spatial Join
crime_with_community = gpd.sjoin(crime_with_ward, community_areas_gdf[['AREA_NUMBE', 'geometry']], how='left', predicate='within', lsuffix='_ward', rsuffix='_community')  # Community Area 매핑

In [14]:
# 8. 'Ward'와 'Community Area' 결측치 채우기
crime_with_community['Ward'] = crime_with_community['Ward'].fillna(crime_with_community['WARD'])
crime_with_community['Community Area'] = crime_with_community['Community Area'].fillna(crime_with_community['AREA_NUMBE'])

In [15]:
# 9. 불필요한 열 삭제 (매핑된 'WARD', 'COMMUNITY' 열 제거)
crime_with_community = crime_with_community.drop(columns=['WARD', 'index_right','index__community', 'AREA_NUMBE'])

In [16]:
crime_with_community = crime_with_community.drop(columns=['geometry'])

In [17]:
# 10. 결측치 확인
print(crime_with_community[['Ward', 'Community Area']].isna().sum())  # 결측치 확인

Ward              2
Community Area    5
dtype: int64


In [18]:
crime_with_community.isna().sum()

,0
Unnamed: 0,0
ID,0
Case Number,0
Date,0
Block,0
IUCR,0
Primary Type,0
Description,0
Location Description,14
Arrest,0


In [19]:
crime_with_community.head()

,Unnamed: 0,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,...,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,0,4673626,HM274058,04/02/2006 01:00:00 PM,055XX N MANGO AVE,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,RESIDENCE,False,...,45.0,11.0,26,1136872.0,1936499.0,2006,04/15/2016 08:55:02 AM,41.981913,-87.771996,"(41.981912692, -87.771996382)"
1,1,4673627,HM202199,02/26/2006 01:40:48 PM,065XX S RHODES AVE,2017,NARCOTICS,MANU/DELIVER:CRACK,SIDEWALK,True,...,20.0,42.0,18,1181027.0,1861693.0,2006,04/15/2016 08:55:02 AM,41.775733,-87.611920,"(41.775732538, -87.611919814)"
2,2,4673628,HM113861,01/08/2006 11:16:00 PM,013XX E 69TH ST,051A,ASSAULT,AGGRAVATED: HANDGUN,OTHER,False,...,5.0,69.0,04A,1186023.0,1859609.0,2006,04/15/2016 08:55:02 AM,41.769897,-87.593671,"(41.769897392, -87.593670899)"
3,4,4673629,HM274049,04/05/2006 06:45:00 PM,061XX W NEWPORT AVE,0460,BATTERY,SIMPLE,RESIDENCE,False,...,38.0,17.0,08B,1134772.0,1922299.0,2006,04/15/2016 08:55:02 AM,41.942984,-87.780057,"(41.942984005, -87.780056951)"
4,5,4673630,HM187120,02/17/2006 09:03:14 PM,037XX W 60TH ST,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,ALLEY,True,...,13.0,65.0,18,1152412.0,1864560.0,2006,04/15/2016 08:55:02 AM,41.784211,-87.716745,"(41.784210853, -87.71674491)"


In [20]:
crime_with_community['Community Area'].unique()

array([11., 42., 69., 17., 65., 67., 66., 24., 32., 35., 45., 25., 63.,
       28., 15., 16.,  8., 68., 76., 73., 12., 41., 50., 39., 10., 30.,
       26., 27., 62., 22., 29.,  6.,  7., 58., 23., 14., 43., 53., 46.,
       71., 52., 34., 44., 51.,  3., 31., 61.,  1., 59., 49., 70., 21.,
       38., 60.,  2., 40., 54., 56., 36., 20., 77., 19., 33., 64., 48.,
       18.,  5.,  4., 47., 57., 55., 37., 72.,  9., 13., 75., 74.,  0.,
       nan])

In [21]:
crime_with_community['Ward'].unique()

array([45., 20.,  5., 38., 13., 17., 15., 32., 42.,  3.,  8., 28., 14.,
        2.,  6., 16., 41., 21., 29., 39.,  4.,  9., 22., 23., 35., 33.,
       24., 44., 43., 12., 37.,  7., 34., 46., 10., 25., 27.,  1., 26.,
       40., 11., 18., 30., 49., 31., 48., 36., 47., 50., 19., nan])

In [22]:
# 11. 결과를 CSV로 저장
crime_with_community.to_csv('/content/drive/MyDrive/chicago_crime_data_with_ward_community2.csv', index=False)


In [ ]:
crime_with_community[['ID', 'Ward', 'Community Area']]

,ID,Ward,Community Area
1,4676906,11.0,61.0
4,4677901,34.0,49.0
6,4791194,9.0,50.0
7,4679521,21.0,73.0
9,4680124,24.0,29.0
...,...,...,...
1923510,4781176,37.0,19.0
1923511,4671197,38.0,15.0
1923512,4671380,18.0,71.0
1923513,4782588,10.0,46.0
